In [ ]:
from pathlib import Path
import numpy as np
import io
from contextlib import redirect_stdout
from tqdm import tqdm
import json
import jax
import re
from datetime import datetime
import h5py
import pandas as pd

# raw to schema
from mangrove.io.metadata import PoseidonMetadata
from mangrove.io.convert.raw_data_to_schema import metadata_to_schema, preprocess_raw_data
from mangrove.beamform.preprocess import preprocess_iq_data_to_rf

# beamforming
from mangrove.beamform.vbeam_ import das_beamformer
from mangrove.io.convert.vbeam.mangrove_to_vbeam import \
    import_space_time_to_vbeam_setup, _time_beamform
from mangrove.schema.wrapper.bmode_wrapper import BModeFile

# power doppler
from mangrove.power_doppler.pca_metal import PCAMetalPowerDoppler

# GLM
from dipy.align import affine_registration
from anise.utils import get_power_doppler_nii, register_image_stack
from anise.process.fusi_glm_fit2 import rat_hrf, fit_glm_time_shift
from nilearn.glm.first_level import make_first_level_design_matrix, FirstLevelModel
from nilearn.image import mean_img

# plotting
from nilearn import plotting
from skimage.measure import label
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.transforms import BlendedGenericTransform

base_path = Path('fUS_data/UCLA/data/UCLA_008/10-23-2024/Functional_runs/run-02/')
sequence = 'seq-IQ_3D_2cmto3cm_Depth_4MHz_1a_3c_200loops_-1Gain_2rp'  # 'seq-IQ_3D_3cmto4cm_Depth_5MHz_1a_3c_200loops_-1Gain_2rp'
"""

sequence = 'seq-IQ_3D_2cmto3cm_Depth_5MHz_1a_3c_200loops_-1Gain_2rp'
"""
experiment_folder = base_path / 'acquisitions' / sequence

event_offset = 0
smoothing_fwhm = 0.3


def decode_event_item(item):
    if isinstance(item, bytes):
        item = item.decode('utf-8')
    if isinstance(item, str) and '{' in item and '}' in item:
        item = json.loads(item)
    return item


# %%
# Load metadata
metadata = PoseidonMetadata.from_folder(experiment_folder / 'metadata')
acquisition_sequence_metadata, transducer_metadata = metadata_to_schema(metadata)
events_raw = pd.DataFrame(columns=['name', 'stimulus', 'time'])
for event in h5py.File(base_path / 'streams' / 'task-event_stream.h5')['data'][:]:
    items = list()
    for item, col in zip(event, events_raw.columns):
        item = decode_event_item(item)
        if isinstance(item, dict) and col in item:
            item = item[col]
        items.append(item)
    events_raw.loc[len(events_raw.index)] = items

events = pd.DataFrame(columns=['onset', 'trial_type', 'duration'])
for i, row in events_raw.iterrows():
    if i % 2:
        continue
    row2 = events_raw.iloc[i + 1]
    assert 'start' in row['name'] or 'on' in row['name']
    assert 'stop' in row2['name'] or 'off' in row2['name']
    assert row['stimulus'] == row2['stimulus']
    duration = row2['time'] - row['time']
    trial_type = row['stimulus'].split('.')[0]
    events.loc[len(events.index)] = row['time'], trial_type, duration


# %%